In [1]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

2026-06-28 15:29:17.704127696 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
v1.dot(dv)

np.float64(0.3233238799303238)

In [3]:
v2.dot(dv)

np.float64(0.019730422395141473)

In [4]:
q = 'How does approximate nearest neighbor search work?'
v = embed.encode(q)

Q1. Embedding a query
The embedder returns a vector of 384 numbers. What's the first value (v[0])?
-0.31
-0.02
0.12
0.44

In [ ]:
v[0]

np.float64(-0.02058203437252893)

In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [7]:
len(documents)

72

In [8]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

Q2. Cosine similarity

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

0.07
0.37
0.68
0.92

In [9]:
result = [x for x in documents if x["filename"]=="02-vector-search/lessons/07-sqlitesearch-vector.md"]
result

[{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done

In [11]:
content = result[0]['content']
c = embed.encode(content)

In [12]:
c.dot(v)

np.float64(0.36107026789538205)

Q3. Chunking and search by hand

Which file does the highest-scoring chunk belong to (its filename)?

02-vector-search/lessons/03-embeddings-dataset.md

02-vector-search/lessons/06-rag-vector.md

02-vector-search/lessons/07-sqlitesearch-vector.md

02-vector-search/lessons/09-onnx-embedder.md

In [13]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [16]:
len(chunks)

295

In [17]:
x = 0
results=[]

for i in chunks:
    if x >= len(chunks):
        break
    else:
        content = chunks[x]['content']
        filename = chunks[x]['filename']
        c = embed.encode(content)
        score = c.dot(v)
        results.append((float(score), filename))
        x +=1

results.sort(reverse=True)
results[0][1]        

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [22]:
texts = []

for doc in documents:
    # text = doc["filename"] + " " + doc["content"]
    text = doc["content"]

    texts.append(text)
   

In [19]:
from tqdm.auto import tqdm

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:02<00:00, 44.23it/s]


In [31]:
batch_size = 20
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

100%|██████████| 4/4 [00:04<00:00,  1.21s/it]


72

In [32]:
import numpy as np
X = np.array(vectors)

In [33]:
X.shape

(72, 384)

In [34]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(X, documents)

In [35]:
query = "I just discovered the course. Can I still join it?"

In [36]:
Q = 'What metric do we use to evaluate a search engine?'

In [38]:
query_vector = model.encode(Q)
results = vindex.search(query_vector, num_results=10)
results

[{'content': "# Evaluation\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=eC_IcxfxoiQ&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous modules, we built search engines and RAG pipelines.\nWe tried different approaches: keyword search with minsearch, vector\nsearch, agents with function calling. But we never answered the obvious\nquestion of which one is actually better.\n\nWe could try a few queries by hand and see what looks good. That's fine\nfor a quick sanity check, but it doesn't scale, and it doesn't give us a\nnumber to compare. We need a systematic way to tell whether one approach\nbeats another.\n\nThat's what evaluation is for. And it's worth saying up front: of\neverything in this course, evaluation is the part that matters most. It's\nalso the most tedious. But it's the only way to be sure your system\nworks. And it's how you keep it working as you change prompts and swap\nmodels.\n\n## The evaluation setup\n\nFor search evaluation, we need a datas

Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

    What metric do we use to evaluate a search engine?

Which file is the filename of the first result?

    02-vector-search/lessons/04-vector-search.md
    04-evaluation/lessons/05-search-metrics.md
    04-evaluation/lessons/13-llm-as-judge.md
    05-monitoring/lessons/04-metrics.md


In [39]:
results[1]

{'content': '# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup, each query 

In [40]:
Q2 = 'How do I store vectors in PostgreSQL?'

In [42]:
query_vector = model.encode(Q2)
results = vindex.search(query_vector, num_results=5)
results

[{'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWORD=pswd \\\n  

In [43]:
from minsearch import Index

index = Index(
    text_fields=["content"]
)

index.fit(documents)

In [47]:
search_results = index.search(
    Q2,
    num_results=5
)

search_results

[{'content': '# Embeddings\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=kJOlW1HeMp4&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nBefore we can do vector search, we need to turn our text into vectors.\nWe call this process embedding: we embed text into a vector space. The\nvectors we get back are also called "embeddings."\n\n## Word embeddings and sentence embeddings\n\nThis idea comes from\n[word2vec](https://en.wikipedia.org/wiki/Word2vec). The model learns to\nplace words as points in a multi-dimensional space. Words with similar\nmeanings land close to each other.\n\nImagine a 2D space where "enroll" and "join" are near each other and\n"Docker" is far away:\n\n```text\n        · enroll\n       · join\n                   · Docker\n```\n\nThe same idea works for entire sentences:\n\n```text\nQ1: "I just discovered the course. Can I still join it?"\nQ2: "I just found out about the program. Can I still enroll?"\n\nThese two are close - they mean the same thing.\n\nQ3: "H

Q5. Text search vs vector search

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

    02-vector-search/lessons/01-intro.md
    02-vector-search/lessons/02-embeddings.md
    02-vector-search/lessons/08-pgvector.md
    03-orchestration/lessons/05-rag.md


##### answer =  02-vector-search/lessons/08-pgvector.md

Q6. Hybrid search

In [48]:
Q3 = "How do I give the model access to tools?"

In [ ]:
text_search_results = index.search(
    Q3
)
text_search_results

[{'content': '# Function Calling\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=CeEki_0mdGo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson we built a RAG pipeline with `RAGBase.rag()`\nand saw it fail on the "Olama" typo. The search returned nothing\nuseful, and the LLM had no way to recover.\n\nThe flow that broke:\n\n```mermaid\nflowchart TD\n    U([User: How do I run Olama?])\n    S[search - Olama - no useful results]\n    A([LLM: I don\'t have information about Olama.])\n\n    U --> S --> A\n```\n\nThe pipeline is fixed: search, build prompt, LLM.\n\n```python\ndef rag(self, query):\n    search_results = self.search(query)\n    prompt = self.build_prompt(query, search_results)\n    answer = self.llm(prompt)\n    return answer\n```\n\nThe LLM is a passenger here, not a driver. It never sees the bad\nsearch results, so it can\'t try again with a corrected query.\n\n## The agent alternative\n\nAn agent puts the LLM in charge.\n\nInstead of running se

In [58]:
query_vector = model.encode(Q3)
vector_search_results = vindex.search(query_vector)
vector_search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [55]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["content"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [60]:
rrf_results = rrf([vector_search_results, text_search_results])
rrf_results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 